# CodeBERT training and energy measurements

Run Notebook 03 first. This notebook trains the frozen `microsoft/codebert-base` embedding model separately on the validated and significant prepared datasets. Inputs are capped at 512 tokens, including tokenizer-added special tokens; pretrained positional embeddings are not extended. Attention-mask mean pooling feeds a class-balanced Logistic Regression head evaluated with five problem-grouped folds.

Training cells measure the complete grouped-training workload, including tokenizer/encoder loading, embedding extraction, grouped model fitting, artifact writes, runtime, process CPU use, peak memory, any supported hardware energy counter, and CodeCarbon estimates when installed. Inference excludes model loading and includes only the sampled embedding and prediction workload. Inference is measured separately on up to 100 held-out pairs for 10 repetitions, then reported per pair and linearly extrapolated to that dataset's pair count. This is an estimate, not a full-dataset inference measurement. If no hardware energy counter is available, direct energy remains unavailable; CodeCarbon values are labeled as estimates rather than measurements. The setup cell creates an external power-meter template for later manual readings.

Run the setup cell once, then both training cells, then the two inference cells. This notebook has not been executed as part of these edits.

In [1]:
import sys
from pathlib import Path

from IPython.display import display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run Notebook 03 first to create Data/Processed.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from measurement_utils import (
    initialize_codebert_resource_experiment,
    measure_codebert_inference,
    train_and_measure_codebert_dataset,
)

MAX_INPUT_LENGTH = 512
BATCH_SIZE = 8
INFERENCE_SAMPLE_PAIRS = 100
INFERENCE_REPETITIONS = 10
PROCESS_SAMPLING_INTERVAL_SECONDS = 0.1

RESOURCE_STATE = initialize_codebert_resource_experiment(
    ROOT,
    sampling_interval_seconds=PROCESS_SAMPLING_INTERVAL_SECONDS,
)
print("Resource run:", RESOURCE_STATE["run_root"])
print("Hardware energy source:", RESOURCE_STATE["energy_counter"].backend or "unavailable")
print("External power-meter template:", RESOURCE_STATE["manual_meter_template"])

Resource run: D:\Habiba\Uni\W26\Bachelor Thesis\energy-aware-code-recommendation\Results\resource_measurements\20261111T133037+0100_967ab8a6
Hardware energy source: unavailable
External power-meter template: D:\Habiba\Uni\W26\Bachelor Thesis\energy-aware-code-recommendation\Results\resource_measurements\20261111T133037+0100_967ab8a6\external_power_meter_manual_entry.csv


In [2]:
validated_performance, validated_training_measurements = train_and_measure_codebert_dataset(
    RESOURCE_STATE,
    dataset_name="validated",
    max_length=MAX_INPUT_LENGTH,
    batch_size=BATCH_SIZE,
)
display(validated_performance)
display(validated_training_measurements)

[codecarbon WARNING @ 13:30:45] Multiple instances of codecarbon are allowed to run at the same time.
d:\Habiba\Uni\W26\Bachelor Thesis\energy-aware-code-recommendation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31887.93it/s]


validated: 21,410 pairs, 40,776 unique code samples; embedding time 11889.5s


,model,feature_config,accuracy_mean,accuracy_std,roc_auc_mean,roc_auc_std,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,f1_std
0,codebert_frozen_embeddings_logistic_regression,codebert_embeddings,0.570668,0.007443,0.61187,0.009798,0.570785,0.007485,0.569679,0.010009


,elapsed_seconds,average_process_cpu_utilization_pct_of_total,baseline_process_memory_mb,peak_process_memory_mb,memory_increase_during_task_mb,cpu_sample_count,memory_sample_count,actual_energy_joules,energy_source,energy_scope,...,task,repetition,measurement_device,encoder_loading_included,training_includes_embedding_extraction,gpu_monitoring_included,n_training_pairs,n_problem_groups,idle_baseline_energy_joules,net_actual_energy_joules
0,11986.883609,50.162528,392.839844,1780.335938,1387.496094,2,114194,None,None,None,...,training,1,cpu,True,True,False,21410,1232,None,None


In [3]:
significant_performance, significant_training_measurements = train_and_measure_codebert_dataset(
    RESOURCE_STATE,
    dataset_name="significant",
    max_length=MAX_INPUT_LENGTH,
    batch_size=BATCH_SIZE,
)
display(significant_performance)
display(significant_training_measurements)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31585.05it/s]


significant: 3,000 pairs, 5,848 unique code samples; embedding time 1449.5s


,model,feature_config,accuracy_mean,accuracy_std,roc_auc_mean,roc_auc_std,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,f1_std
0,codebert_frozen_embeddings_logistic_regression,codebert_embeddings,0.641333,0.030354,0.702407,0.040327,0.64239,0.029961,0.649879,0.033479


,elapsed_seconds,average_process_cpu_utilization_pct_of_total,baseline_process_memory_mb,peak_process_memory_mb,memory_increase_during_task_mb,cpu_sample_count,memory_sample_count,actual_energy_joules,energy_source,energy_scope,...,task,repetition,measurement_device,encoder_loading_included,training_includes_embedding_extraction,gpu_monitoring_included,n_training_pairs,n_problem_groups,idle_baseline_energy_joules,net_actual_energy_joules
0,1455.955079,49.868694,475.816406,1148.023438,672.207031,2,13858,None,None,None,...,training,1,cpu,True,True,False,3000,476,None,None


## Sampled inference measurements

Each following cell reuses the saved held-out fold heads, encodes the same deterministic sample of up to 100 pairs, and repeats inference 10 times. The recorded per-pair energy and dataset-scale energy are linear extrapolations from those measured batches. Model/encoder loading is excluded; the encoder embedding and Logistic Regression prediction are included.

In [4]:
validated_inference_measurements = measure_codebert_inference(
    RESOURCE_STATE,
    dataset_name="validated",
    inference_pairs=INFERENCE_SAMPLE_PAIRS,
    inference_repetitions=INFERENCE_REPETITIONS,
    batch_size=BATCH_SIZE,
)
display(validated_inference_measurements.query("task == 'inference'"))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24559.85it/s]


,elapsed_seconds,average_process_cpu_utilization_pct_of_total,baseline_process_memory_mb,peak_process_memory_mb,memory_increase_during_task_mb,cpu_sample_count,memory_sample_count,actual_energy_joules,energy_source,energy_scope,...,encoder_loading_included,gpu_monitoring_included,gross_actual_energy_joules_per_pair,idle_baseline_energy_joules,net_actual_energy_joules,net_actual_energy_joules_per_pair,codecarbon_estimated_energy_joules_per_pair,extrapolated_gross_actual_energy_joules,extrapolated_net_actual_energy_joules,extrapolated_codecarbon_estimated_energy_joules
0,53.578371,49.985731,967.671875,1192.054688,224.382812,2,510,None,None,None,...,False,False,None,None,None,None,13.402440,None,None,286946.235572
1,55.221462,50.060143,1087.125000,1206.886719,119.761719,2,527,None,None,None,...,False,False,None,None,None,None,13.924946,None,None,298133.083307
2,51.067610,49.947932,1089.531250,1208.832031,119.300781,2,488,None,None,None,...,False,False,None,None,None,None,12.904153,None,None,276277.909432
3,51.542773,49.949688,1089.644531,1192.980469,103.335938,2,492,None,None,None,...,False,False,None,None,None,None,12.993852,None,None,278198.369766
4,51.874200,49.978835,1087.765625,1202.968750,115.203125,2,496,None,None,None,...,False,False,None,None,None,None,13.171748,None,None,282007.115807
5,51.535009,50.014171,1086.636719,1206.718750,120.082031,2,492,None,None,None,...,False,False,None,None,None,None,13.174388,None,None,282063.655781
6,49.701185,49.988916,1100.269531,1192.652344,92.382812,2,474,None,None,None,...,False,False,None,None,None,None,11.870772,None,None,254153.226466
7,49.074467,50.006760,1081.539062,1201.652344,120.113281,2,468,None,None,None,...,False,False,None,None,None,None,12.281702,None,None,262951.233279
8,49.505997,50.041979,1084.261719,1201.011719,116.750000,2,472,None,None,None,...,False,False,None,None,None,None,12.369049,None,None,264821.336564
9,49.496311,50.032342,1084.191406,1201.406250,117.214844,2,474,None,None,None,...,False,False,None,None,None,None,12.516605,None,None,267980.517852


In [5]:
significant_inference_measurements = measure_codebert_inference(
    RESOURCE_STATE,
    dataset_name="significant",
    inference_pairs=INFERENCE_SAMPLE_PAIRS,
    inference_repetitions=INFERENCE_REPETITIONS,
    batch_size=BATCH_SIZE,
)
display(significant_inference_measurements.query("task == 'inference'"))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28243.99it/s]


,elapsed_seconds,average_process_cpu_utilization_pct_of_total,baseline_process_memory_mb,peak_process_memory_mb,memory_increase_during_task_mb,cpu_sample_count,memory_sample_count,actual_energy_joules,energy_source,energy_scope,...,encoder_loading_included,gpu_monitoring_included,gross_actual_energy_joules_per_pair,idle_baseline_energy_joules,net_actual_energy_joules,net_actual_energy_joules_per_pair,codecarbon_estimated_energy_joules_per_pair,extrapolated_gross_actual_energy_joules,extrapolated_net_actual_energy_joules,extrapolated_codecarbon_estimated_energy_joules
0,62.112457,49.940144,1148.199219,1382.589844,234.390625,2,592,None,None,None,...,False,False,None,None,None,None,16.004523,None,None,48013.567527
1,59.170564,50.003817,1279.988281,1377.121094,97.132812,2,566,None,None,None,...,False,False,None,None,None,None,15.616758,None,None,46850.274855
2,61.055574,50.014395,1291.484375,1371.078125,79.593750,2,581,None,None,None,...,False,False,None,None,None,None,17.304224,None,None,51912.672961
3,60.831567,50.007600,1231.832031,1371.046875,139.214844,2,580,None,None,None,...,False,False,None,None,None,None,17.273840,None,None,51821.518768
4,59.246166,49.998802,1242.535156,1374.300781,131.765625,2,565,None,None,None,...,False,False,None,None,None,None,15.834042,None,None,47502.126548
5,56.418289,49.991761,1293.152344,1379.066406,85.914062,2,540,None,None,None,...,False,False,None,None,None,None,14.727022,None,None,44181.066402
6,56.689069,50.064792,1240.894531,1369.058594,128.164062,2,542,None,None,None,...,False,False,None,None,None,None,15.088141,None,None,45264.421789
7,57.339745,50.051780,1239.257812,1377.441406,138.183594,2,548,None,None,None,...,False,False,None,None,None,None,14.909640,None,None,44728.920852
8,57.910729,50.013549,1271.132812,1369.433594,98.300781,2,554,None,None,None,...,False,False,None,None,None,None,15.103906,None,None,45311.718867
9,63.817718,49.979778,1239.261719,1361.578125,122.316406,2,608,None,None,None,...,False,False,None,None,None,None,17.614701,None,None,52844.102267
